# TSMixer: Перемешивание вместо внимания

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/13_tsmixer.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q neuralforecast datasetsforecast pandas numpy matplotlib seaborn

## Подготовка многомерных данных

In [ ]:
import pandas as pd
import numpy as np

# Создаём многомерные синтетические данные
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')

# Несколько связанных рядов
base = np.cumsum(np.random.randn(365))
series_data = []

for i in range(5):
    y = 100 + base + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi + i) + np.random.randn(365) * 5
    series_data.append(pd.DataFrame({
        'unique_id': f'series_{i}',
        'ds': dates,
        'y': y
    }))

train = pd.concat(series_data, ignore_index=True)
print(f"Всего рядов: {train['unique_id'].nunique()}")
print(train.head(10))

## TSMixer: конфигурация и обучение

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import TSMixer
from neuralforecast.losses.pytorch import MAE

# Параметры
HORIZON = 16
INPUT_SIZE = 64  # длина входного окна

# Конфигурация TSMixer
model = TSMixer(
    h=HORIZON,
    input_size=INPUT_SIZE,
    n_series=None,                    # автоматически определится из данных
    loss=MAE(),
    max_steps=1000,
    
    # Архитектура mixer
    n_block=4,                        # количество mixer-блоков
    ff_dim=64,                        # размерность feed-forward слоёв
    dropout=0.1,
    
    scaler_type='standard',
    random_seed=42
)

# Обучаем
nf = NeuralForecast(
    models=[model],
    freq='D'
)
nf.fit(df=train)

# Прогнозируем
forecasts = nf.predict()
print(forecasts)

## Функция подготовки многомерных данных

In [ ]:
def prepare_multivariate_data(df, group_col, time_col, value_col, channel_col):
    """
    Преобразует данные в формат, подходящий для TSMixer.
    
    Args:
        df: исходный DataFrame
        group_col: колонка с группировкой (например, store_id)
        time_col: колонка с датой
        value_col: колонка со значениями
        channel_col: колонка с каналами (например, category)
    
    Returns:
        DataFrame в long format с unique_id = group_channel
    """
    df = df.copy()
    
    # Создаём составной идентификатор
    df['unique_id'] = df[group_col].astype(str) + '_' + df[channel_col].astype(str)
    
    # Переименовываем колонки
    df = df.rename(columns={time_col: 'ds', value_col: 'y'})
    
    # Оставляем только нужные колонки
    df = df[['unique_id', 'ds', 'y']]
    
    return df

# Пример использования (закомментировано)
# train_prepared = prepare_multivariate_data(
#     raw_data,
#     group_col='store_id',
#     time_col='date',
#     value_col='sales',
#     channel_col='category'
# )

## Сравнение TSMixer с N-HiTS

In [ ]:
from neuralforecast.models import NHITS

# N-HiTS для сравнения
model_nhits = NHITS(
    h=HORIZON,
    input_size=INPUT_SIZE,
    loss=MAE(),
    max_steps=1000,
    n_pool_kernel_size=[1, 2, 4],
    n_freq_downsample=[1, 2, 4],
    scaler_type='standard',
    random_seed=42
)

# TSMixer
model_tsmixer = TSMixer(
    h=HORIZON,
    input_size=INPUT_SIZE,
    loss=MAE(),
    max_steps=1000,
    n_block=4,
    ff_dim=64,
    scaler_type='standard',
    random_seed=42
)

# Обучаем обе модели
nf_compare = NeuralForecast(
    models=[model_nhits, model_tsmixer],
    freq='D'
)
nf_compare.fit(df=train)

# Сравниваем результаты
forecasts_compare = nf_compare.predict()
print(forecasts_compare)

## Визуализация корреляций между каналами

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Корреляции в обучающих данных
train_wide = train.pivot(index='ds', columns='unique_id', values='y')
train_corr = train_wide.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(train_corr, ax=ax, cmap='coolwarm', center=0,
            annot=True, fmt='.2f')
ax.set_title('Корреляции между рядами в обучающих данных')
plt.tight_layout()
plt.show()

## Визуализация прогнозов

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Выбираем один ряд для визуализации
sample_uid = 'series_0'
history = train[train['unique_id'] == sample_uid].tail(50)
forecast_data = forecasts_compare[forecasts_compare['unique_id'] == sample_uid]

# TSMixer
axes[0].plot(history['ds'], history['y'], label='История', color='blue')
axes[0].plot(forecast_data['ds'], forecast_data['TSMixer'], 
             label='TSMixer', linestyle='--', color='red')
axes[0].set_title('TSMixer прогноз')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# N-HiTS
axes[1].plot(history['ds'], history['y'], label='История', color='blue')
axes[1].plot(forecast_data['ds'], forecast_data['NHITS'], 
             label='N-HiTS', linestyle='--', color='green')
axes[1].set_title('N-HiTS прогноз')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()